# Extract CONCH (VLM) image + text features

Run once per (dataset, seed, VLM, description style). Unlike DINOv2, a VLM has
**two image embedding spaces** that are not interchangeable
(`features/vlm.py` module docstring has the full detail):

* `RAW_SPACE` (`proj_contrast=False, normalize=False`) -- for the linear probe,
  the coverage kernel, everything `scalpel`'s disagreement machinery reads.
* `PROJ_SPACE` (`proj_contrast=True, normalize=True`) -- for comparing an
  image against text, which is what the round-1 cold-start prior needs.

Both are written from **one trunk pass per batch**: RAW is CONCH's
`forward_no_head` output directly, and PROJ is derived from it with one
matmul (`raw @ visual.proj_contrast`, then L2-normalize) rather than by
calling `encode_image` a second time. Verified bit-exact against calling it
twice (`tests/test_vlm_features.py`) — the two spaces share every step up to
the pooled vector, so recomputing the whole 12-layer ViT-B trunk on 448x448
pixels (4x DINOv2's pixel count) for PROJ alone would double this notebook's
most expensive step for no numerical difference.

This also builds the **text prototypes**: one 512-d vector per class, either
from the official CONCH CRC100K prompt set (`conch_official` -- vendored in
`config/prompts/`, PathMNIST only) or from a description file
`generate_class_description.ipynb` wrote. `manual` reads
`datasets.<name>.descriptions` from `config.yaml` directly, no file needed.

**One configuration per run** (`DATASET`, `SEED`, `VLM`, `DESCRIPTION_STYLE` are
all singular): the notebook ends in one zip whose name states the whole
configuration. Sweeping a style means running the notebook again.

**Both T4s are used.** Work is split by contiguous row range, one worker
process per GPU, then assembled into the single cache layout
`run_al_main.ipynb` reads. Because the split is contiguous and the backbone is
frozen, the assembled cache is row-for-row identical to a one-GPU run —
`tests/test_vlm_shards.py` asserts exactly that, for BOTH spaces. Set
`PARALLEL = False` for the single-GPU reference path.

**`MMAP_CACHE_DIR` is required for a 2-GPU run on a `.npz` dataset.** The
limit is RAM, not VRAM: features return to CPU each batch, so a GPU holds one
batch, but an eager `.npz` read costs ~15 GiB *per process* and two workers
then exceed the ~30 GiB a Kaggle session has. The symptom is a worker that
prints nothing at all (killed inside `np.load`) while the other runs on. The
parent exports the `.npz` to memory-mappable `.npy` once, before the workers
start; `numpy` silently ignores `mmap_mode` for a `.npz`, so only that export
can be mapped.

**Two Kaggle Datasets required**, same reasoning as `run_al_baseline.ipynb`:
the raw images (`DATA_ROOT`) and nothing else -- there is no VLM cache to
attach yet on a first run, since this notebook is what produces one.

Zero-shot accuracy on the pool, using whichever text prototype this run built,
is printed and asserted `> 0.70` at the end -- not to reproduce the paper's
79.1% (PathMNIST is 224-native, resized up to CONCH's 448, so an exact match
is not expected), but to catch a wrong transform, normalization, tokenizer or
projection before it silently corrupts every cold-start result built on this
cache. A near-11%-random score means something upstream is wrong.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])
# open_clip_torch is NOT enough: CONCH has its own tokenizer + factory
# (features/vlm.py module docstring, section on the tokenizer). Install the
# conch package itself.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "git+https://github.com/mahmoodlab/CONCH.git"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"
SEED = 42                          # ONE seed, ONE zip

# CONCH | Astaxanthin/KEEP | wisdomik/QuiltNet-B-16-PMB | vinid/plip
# Only CONCH is verified against the paper/code in PLAN_IMPLEMENT.md §10 --
# a different VLM likely needs its own loader in features/vlm.py.
VLM = "MahmoodLab/CONCH"

# manual | conch_official | llm_short | llm_morphology
#   manual         : datasets.<DATASET>.descriptions in config.yaml, no file
#   conch_official : the paper authors' own 22-template x 4-5-classname
#                     ensemble for CRC100K -- PathMNIST ONLY, vendored in
#                     config/prompts/. The baseline any LLM description must
#                     beat to justify itself as a contribution.
#   llm_*          : written by generate_class_description.ipynb; must have
#                     been run first, or this notebook fails asking for it.
DESCRIPTION_STYLE = "conch_official"

HF_TOKEN = ""                      # CONCH is gated on HF -- needed to download it

# Two Kaggle Datasets, same split as run_al_baseline.ipynb: the raw images
# (labels + fingerprint) are required even though this notebook is what
# BUILDS the VLM cache -- there is nothing to attach for that side yet.
DATA_ROOT = "/kaggle/input/datasets/cryandrrich/nckh2026"
FEATURE_DIR = "/kaggle/working/vlm_features"

# Use both T4s: work is split by contiguous row range, one worker process per
# GPU, then assembled into the single cache layout run_al_main.ipynb reads.
# False forces one GPU, which is the reference path the parallel one is tested
# against (tests/test_vlm_shards.py asserts they are row-for-row identical).
PARALLEL = True

# VRAM knob. 448x448 is 4x DINOv2's pixel count, and a shard worker holds the
# CONCH checkpoint plus one batch of activations on a 16 GiB T4. 64 is the
# tested value; lower it if a worker hits CUDA OOM (the symptom is an explicit
# CUDA error, unlike the RAM problem below, which kills a process silently).
BATCH_SIZE = 64

# Where the .npz is re-exported as memory-mappable .npy files. REQUIRED for a
# 2-GPU run on PathMNIST, and the reason is RAM, not VRAM: an eager .npz read
# costs ~15 GiB *per process*, so two workers exceed the ~30 GiB a Kaggle
# session has and one is OOM-killed inside np.load before printing anything.
# Needs about as much scratch disk as the .npz itself (~15 GiB for
# pathmnist_224). Ignored for ImageFolder datasets, which read per file.
MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

In [ ]:
from huggingface_hub import login

# CONCH is a GATED repo on Hugging Face (`gated: auto`), so this notebook --
# the only one that actually downloads the checkpoint -- cannot run without
# a token tied to an account that has ACCEPTED the license at
# huggingface.co/MahmoodLab/CONCH. Accepting is a one-time click; a token
# alone is not enough, and a non-accepted account gets 401/403 no matter how
# valid its token is.
#
# Resolve the same way run_al_main.ipynb and generate_class_description.ipynb
# do: an explicit HF_TOKEN first, then a Kaggle Secret. An earlier version of
# this cell only PRINTED that it was "relying on a Kaggle Secret" and never
# read one -- so a user who had set the Secret up correctly, following the
# message, still failed at the download cell with no indication why.
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("[auth] using Kaggle Secret HF_TOKEN")
    except Exception as exc:
        print(f"[auth] no Kaggle Secret HF_TOKEN ({type(exc).__name__})")

assert HF_TOKEN, (
    "CONCH is gated on Hugging Face and this notebook downloads it, so a "
    "token is required. Either set HF_TOKEN in the EDIT cell, or add a "
    "Kaggle Secret named HF_TOKEN (Add-ons -> Secrets). Also accept the "
    "license once at https://huggingface.co/MahmoodLab/CONCH -- a valid "
    "token for an account that has not accepted still gets 401/403."
)

login(HF_TOKEN)
# Put it in the environment too: the sharded extraction below spawns one
# worker process per GPU, and those inherit os.environ but not this cell's
# local variables.
import os

os.environ["HF_TOKEN"] = HF_TOKEN
print("[auth] logged in to the HF Hub; CONCH checkpoint will download on first use")


In [ ]:
import json

import numpy as np
import yaml
import torch

from data.loaders import get_data_loaders, get_sample_ids
from data.identity import sample_order_fingerprint
from data.npz_mmap import export_npz_to_npy
from features.vlm import (
    RAW_SPACE,
    PROJ_SPACE,
    assemble_vlm_feature_shards,
    assert_class_order_matches_prompts,
    description_sha256,
    encode_text_prototypes,
    get_or_extract_vlm_features,
    load_conch,
    load_official_conch_prompts,
    text_prototype_cache_paths,
    vlm_feature_cache_paths,
    zero_shot_logits,
)
from scripts.extract_vlm_features import build_vlm_shard_jobs, extract_vlm_shard_on_worker
from utils import vlm_archive_stem
from utils.kaggle import find_data_root
from utils.parallel import run_variants_parallel, visible_gpu_count

# The same label-extraction helper main.run() uses -- imported here (not in
# the zero-shot cell below) because the parent reads test labels while it
# still holds the loader, before the workers start.
import main as _main

In [ ]:
DATA_ROOT_RESOLVED = find_data_root([Path(DATA_ROOT)])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT_RESOLVED / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT_RESOLVED / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT_RESOLVED / "SkinTissue/SkinTissue/tiles"),
}
print("data root (raw images):", DATA_ROOT_RESOLVED)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before extraction"
assert not str(FEATURE_DIR).startswith("/kaggle/input"), (
    "FEATURE_DIR must be writable; /kaggle/input is read-only"
)
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

# class_names in config.yaml order -- the order the probe/coverage side of the
# pipeline already uses everywhere else, so the text prototypes below are
# built in that same order and nothing needs a translation table.
dataset_info = config["datasets"][DATASET]
class_names = list(dataset_info["descriptions"])
num_classes = dataset_info["num_classes"]
assert len(class_names) == num_classes, (
    f"config.yaml lists {len(class_names)} descriptions but "
    f"num_classes={num_classes} for {DATASET!r}"
)

# Resolve DESCRIPTION_STYLE into a (class_names -> list-of-prompt-strings) map,
# built once here so the extraction cell below is style-agnostic.
if DESCRIPTION_STYLE == "manual":
    class_prompts = [[dataset_info["descriptions"][name]] for name in class_names]
    description_source = "config.yaml datasets.<dataset>.descriptions"
    description_hash = description_sha256(dataset_info["descriptions"])

elif DESCRIPTION_STYLE == "conch_official":
    assert DATASET == "pathmnist", (
        "the official CONCH prompt set is CRC100K-specific (9 classes) and "
        f"only matches PathMNIST's classes, not {DATASET!r}"
    )
    prompts = load_official_conch_prompts("config/prompts/crc100k_prompts_all_per_class.json")
    assert_class_order_matches_prompts(class_names, prompts["classnames"])
    # Official ensemble: every (classname x template) pair, per class -- see
    # features/vlm.py::encode_text_prototypes for why this must not be
    # collapsed to one string per class before encoding.
    templates = prompts["templates"]
    class_prompts = [
        [template.replace("CLASSNAME", classname)
         for classname in prompts["classnames"][code]
         for template in templates]
        for code in prompts["classnames"]
    ]
    description_source = "config/prompts/crc100k_prompts_all_per_class.json (official CONCH prompts)"
    description_hash = description_sha256(prompts["classnames"])

else:
    # llm_short | llm_morphology -- written by
    # generate_class_description.ipynb into config/descriptions/.
    description_path = Path(f"config/descriptions/{DATASET}_{DESCRIPTION_STYLE}.json")
    assert description_path.is_file(), (
        f"No description file at {description_path}. Run "
        "generate_class_description.ipynb with this DATASET/STYLE first."
    )
    with open(description_path, "r", encoding="utf-8") as handle:
        description_payload = json.load(handle)
    assert list(description_payload["descriptions"]) == class_names, (
        f"{description_path} class order does not match config.yaml's -- "
        "regenerate it against the current config"
    )
    class_prompts = [[description_payload["descriptions"][name]] for name in class_names]
    description_source = str(description_path)
    description_hash = description_payload.get("sha256") or description_sha256(
        description_payload["descriptions"]
    )

print(f"VLM: {VLM} | description_style: {DESCRIPTION_STYLE}")
print(f"description source: {description_source}")
print(f"classes ({len(class_names)}): {class_names}")
SHARDS = visible_gpu_count() if PARALLEL else 1
SHARDS = max(1, SHARDS)

# RAM, not VRAM, is what limits a 2-GPU extraction: features go back to CPU
# each batch, so the GPU footprint is one batch -- the *pixels* are the
# problem. Report the budget so an OOM restart is diagnosable from the log
# instead of a silent worker death.
import shutil as _shutil

total_npz = data_path.stat().st_size if str(data_path).endswith(".npz") else 0
free_disk = _shutil.disk_usage("/kaggle/working").free if Path("/kaggle/working").exists() \
    else _shutil.disk_usage(".").free
print(f"GPUs visible: {visible_gpu_count()} | shards: {SHARDS} | batch: {BATCH_SIZE}")
if total_npz:
    print(f"npz input: {total_npz / 2**30:.1f} GiB | mmap export needs about the same")
    print(f"free disk: {free_disk / 2**30:.1f} GiB at {MMAP_CACHE_DIR}")
    assert free_disk > total_npz * 1.1, (
        f"Not enough scratch disk for the .npy export: need ~{total_npz / 2**30:.1f} GiB, "
        f"have {free_disk / 2**30:.1f} GiB."
    )
try:
    with open("/proc/meminfo") as handle:
        mem_total = int(next(l for l in handle if l.startswith("MemTotal")).split()[1]) * 1024
    print(f"system RAM: {mem_total / 2**30:.1f} GiB across {SHARDS} worker(s)")
except (OSError, StopIteration, ValueError):
    pass

In [ ]:
import time

# Both T4s. Work is split by CONTIGUOUS row range, one worker process per GPU,
# then assembled into the single cache layout run_al_main.ipynb already reads.
# Contiguous (not round-robin) because a strided split would put different
# samples in a batch than a serial run does; the forward pass here is
# batch-independent, but keeping the split contiguous is what makes the
# assembled cache comparable to a serial one row by row -- which is exactly
# how tests/test_vlm_shards.py pins it.
started = time.time()
device = torch.device("cuda:0")
print(f"{DATASET} | seed {SEED} | {VLM} | shards={SHARDS}")

vlm_paths = vlm_feature_cache_paths(FEATURE_DIR, DATASET, SEED, VLM)

# Export the .npz to memory-mappable .npy ONCE, here in the PARENT.
#
# This is what keeps the session inside its RAM budget. An eager .npz read
# costs ~15 GiB for PathMNIST-224 -- and numpy silently IGNORES mmap_mode for
# a .npz (members go through zipfile), so only a standalone .npy can be
# mapped. Two workers each holding an eager copy exceed the ~30 GiB Kaggle
# gives you and one is OOM-killed inside np.load, before printing a line.
#
# It must happen in the parent: two workers exporting the same files
# concurrently would race. Once exported, every process maps the same pages
# and the OS page cache serves them all from one copy, so a second worker
# costs no additional pixel memory.
mmap_dir = None
if str(data_path).endswith(".npz"):
    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    mmap_dir = MMAP_CACHE_DIR
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

# Fingerprints come from the PARENT so the manifest describes the split the
# workers actually sharded. Built with CONCH's transform is unnecessary here
# (sample IDs and labels do not depend on pixels), so this loader is cheap.
train_loader, test_loader, _ = get_data_loaders(
    str(data_path), SEED, verbose=True, mmap_cache_dir=mmap_dir,
)
n_train, n_test = len(train_loader.dataset), len(test_loader.dataset)
train_fingerprint = sample_order_fingerprint(get_sample_ids(train_loader.dataset))
test_fingerprint = sample_order_fingerprint(get_sample_ids(test_loader.dataset))
test_labels = _main._dataset_labels(test_loader.dataset)
print(f"  train={n_train} test={n_test} mmap={getattr(train_loader.dataset, 'mmap', None)}")

# Release the parent's own loader handles BEFORE the workers start: whatever
# the parent still holds is RAM the two children cannot use.
del train_loader, test_loader

if SHARDS == 1:
    # Reference path: one GPU, no sharding. This is what the parallel path is
    # tested against, and what a PARALLEL=False run takes.
    conch_model, conch_preprocess = load_conch(VLM, device, hf_token=HF_TOKEN)
    train_loader, test_loader, _ = get_data_loaders(
        str(data_path), SEED, mmap_cache_dir=mmap_dir, transform=conch_preprocess,
    )
    # get_data_loaders' own batch size (256) assumes DINOv2's 224x224; CONCH's
    # 448x448 is 4x the pixels, so rebuild at BATCH_SIZE.
    train_loader = torch.utils.data.DataLoader(
        train_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=train_loader.num_workers, pin_memory=True,
    )
    test_loader = torch.utils.data.DataLoader(
        test_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=test_loader.num_workers, pin_memory=True,
    )
    cached = get_or_extract_vlm_features(
        train_loader, test_loader, DATASET, SEED, VLM, device,
        cache_dir=FEATURE_DIR,
        train_fingerprint=train_fingerprint, test_fingerprint=test_fingerprint,
        model=conch_model, hf_token=HF_TOKEN,
    )
    del train_loader, test_loader
elif Path(vlm_paths["manifest"]).is_file() and Path(vlm_paths["proj_manifest"]).is_file():
    # A finished cache from an earlier session: nothing to extract. The model
    # is still needed below for the text prototypes, so load it here.
    print("complete cache exists, skipping extraction")
    cached = {k: np.load(vlm_paths[k]) for k in ("train", "test", "proj_train", "proj_test")}
    conch_model, _ = load_conch(VLM, device, hf_token=HF_TOKEN)
else:
    # Each worker loads its OWN checkpoint: a CUDA-initialised module does not
    # survive spawn pickling, and the parent must not touch CUDA before
    # forking. The download already happened in the auth cell above, so this
    # is a local read from the shared HF cache, not a second download.
    jobs = build_vlm_shard_jobs(
        str(data_path), DATASET, SEED, VLM, SHARDS, FEATURE_DIR,
        batch_size=BATCH_SIZE, mmap_cache_dir=mmap_dir, hf_token=HF_TOKEN or None,
    )
    results = run_variants_parallel(jobs, extract_vlm_shard_on_worker, num_workers=SHARDS)
    for result in results:
        status = "ok" if result["ok"] else "FAILED"
        print(f"  {result['label']:36} {status}")
    failed = [r["label"] for r in results if not r["ok"]]
    assert not failed, f"shards failed: {failed}"

    # Assembly writes both manifests LAST, so a cache is only trusted once
    # every shard is present and its row counts check out.
    cached = assemble_vlm_feature_shards(
        DATASET, SEED, VLM, SHARDS, n_train, n_test, cache_dir=FEATURE_DIR,
        train_fingerprint=train_fingerprint, test_fingerprint=test_fingerprint,
    )
    # The text prototypes below need the model, and no worker's copy survives
    # its process. One load in the parent, after the GPUs are free again.
    conch_model, _ = load_conch(VLM, device, hf_token=HF_TOKEN)

print(f"raw:  train {cached['train'].shape} | test {cached['test'].shape}")
print(f"proj: train {cached['proj_train'].shape} | test {cached['proj_test'].shape}")
print(f"total {time.time() - started:.0f}s")


In [ ]:
# Text prototypes: one 512-d vector per class, in PROJ_SPACE (the space image
# embeddings are compared against). Cached separately from the image features
# -- see features/vlm.py::text_prototype_cache_paths -- because it depends on
# (dataset, style) only, not on seed or the train/test split.
text_paths = text_prototype_cache_paths(FEATURE_DIR, DATASET, DESCRIPTION_STYLE)

if Path(text_paths["prototypes"]).is_file() and Path(text_paths["manifest"]).is_file():
    with open(text_paths["manifest"], "r", encoding="utf-8") as handle:
        text_manifest = json.load(handle)
    prototypes_match = (
        text_manifest.get("class_names") == class_names
        and text_manifest.get("description_sha256") == description_hash
    )
    if prototypes_match:
        text_prototypes = np.load(text_paths["prototypes"])
        print(f"[text] Loaded cache -> {text_paths['prototypes']} {text_prototypes.shape}")
    else:
        print("[text] Cache exists but class order or description changed -- recomputing.")
else:
    prototypes_match = False

if not prototypes_match:
    text_prototypes_t = encode_text_prototypes(conch_model, class_names, class_prompts, device)
    text_prototypes = text_prototypes_t.cpu().numpy().astype(np.float32)
    os.makedirs(FEATURE_DIR, exist_ok=True)
    np.save(text_paths["prototypes"], text_prototypes)
    with open(text_paths["manifest"], "w", encoding="utf-8") as handle:
        json.dump({
            "dataset": DATASET,
            "style": DESCRIPTION_STYLE,
            "vlm": VLM,
            "class_names": class_names,
            "description_source": description_source,
            "description_sha256": description_hash,
            "prompts_per_class": [len(p) for p in class_prompts],
        }, handle, indent=2, sort_keys=True)
    print(f"[text] Saved -> {text_paths['prototypes']} {text_prototypes.shape}")

# LEARNED, not the log(1/0.07) init value -- training moves it, and the
# official zero-shot code reads it off the loaded checkpoint every time
# (features/vlm.py module docstring, logit_scale section). Captured here,
# before the model is freed, since nothing after this needs the model itself.
conch_logit_scale = conch_model.logit_scale.exp().item()

del conch_model
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Zero-shot check: not a reproduction of the paper's 79.1% (PathMNIST is
# 224-native, resized up to CONCH's 448 here, so an exact match is not
# expected -- see PLAN_IMPLEMENT.md 4.2), but a check that transform,
# normalization, tokenizer and projection are all correct. A wrong one of
# those does not crash; it silently produces near-random accuracy (~11% for
# 9 classes) with no other symptom.
# test_labels was read in the extraction cell, while the parent still held
# the loader -- the workers do not return one and `test_loader` is deleted
# before they start.
probs = zero_shot_logits(cached["proj_test"], text_prototypes, logit_scale=conch_logit_scale)
predictions = probs.argmax(axis=1)
zero_shot_accuracy = float((predictions == np.asarray(test_labels)).mean())

print(f"zero-shot accuracy on {DATASET} test ({len(test_labels)} images): "
      f"{zero_shot_accuracy:.4f}")
assert zero_shot_accuracy > 0.70, (
    f"zero-shot accuracy {zero_shot_accuracy:.4f} is far below the ~0.79 the "
    "paper reports on CRC100K -- check transform/normalization/tokenizer/"
    "projection before trusting this cache for anything downstream. A score "
    f"near 1/{num_classes} = {1 / num_classes:.3f} means something upstream "
    "is silently wrong, not that this dataset is simply harder."
)

In [ ]:
# Verify what will be shipped: both spaces present, complete, finite, and
# consistent with the manifests. Cheaper to fail here than after the zip has
# been published and attached to a run notebook.
vlm_paths = vlm_feature_cache_paths(FEATURE_DIR, DATASET, SEED, VLM)
for split in ("train", "test", "proj_train", "proj_test"):
    array = np.load(vlm_paths[split], mmap_mode="r")
    assert np.all(np.isfinite(array[:256])), f"{split} features are not all finite"
with open(vlm_paths["manifest"], "r", encoding="utf-8") as handle:
    manifest = json.load(handle)
with open(vlm_paths["proj_manifest"], "r", encoding="utf-8") as handle:
    proj_manifest = json.load(handle)
assert manifest["space"] == RAW_SPACE and proj_manifest["space"] == PROJ_SPACE
assert manifest["dataset"] == DATASET and manifest["seed"] == SEED
assert manifest["backbone"] == VLM

assert Path(text_paths["prototypes"]).is_file()
assert text_prototypes.shape == (num_classes, text_prototypes.shape[1])
print(f"OK {DATASET} seed{SEED} {VLM}: image {cached['train'].shape}/{cached['proj_train'].shape}, "
      f"text {text_prototypes.shape}, zero-shot acc {zero_shot_accuracy:.4f}")

In [ ]:
# Package the cache as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape every other publishing notebook in this
# project uses.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI. Keeping the originals beside
# the zip also doubles the download, and a session over the ~20 GB Output
# quota shows NOTHING at all, including the files that were fine.
import shutil

# Delete the .npy mmap export BEFORE archiving.
#
# It is scratch for the pixel-reading pass only -- nothing below it, and
# nothing downstream, ever reads it again. But it is the single largest thing
# in /kaggle/working: PathMNIST-224 exports ~15 GiB, and a real run of this
# notebook finished with 16.1 GB sitting in Output, ~15.7 GB of it this
# directory. That is 80% of the ~20 GB Output quota spent on a temporary
# file, and a session that goes over the quota shows NOTHING in the Output
# tab -- including the zip that was fine. Same failure the nucleus run hit
# (CLAUDE.md: "The nucleus run dies of DISK, not memory").
if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")
SOURCE = Path(FEATURE_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "FEATURE_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = vlm_archive_stem(DATASET, SEED, VLM, DESCRIPTION_STYLE)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.iterdir()):
    print(f"    {path.name}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
     Kaggle extracts it into a directory named after the zip, so the cache
     files end up one level down. That is expected.
  3. In run_al_main.ipynb: Add Data -> your new dataset. The VLM feature
     cache is resolved by filename, so there is no path to edit.""")